# 05. Structured Output (Advanced) — 제품 탑재형 AI: 기기 제어/진단 요청서 파싱
**Pydantic 스키마 + Cross-field Validation + Discriminated Union + Repair(재생성) 전략**

---

## 🧩 시나리오(가상)
제품에 탑재된 LLM이 사용자의 자연어를 바로 실행하면 위험합니다.  
대신, **반드시 구조화된 JSON(명령/진단/질문)** 으로 변환한 뒤에

- 안전 체크(확인/권한)
- 실행 가능한 API/SDK 호출
- 로그/근거 저장

같은 파이프라인을 거칩니다.

이 노트북은 그중 핵심인 **"자연어 → 스키마(JSON)"** 를 튼튼하게 만드는 실습입니다.

> **사전 조건 (로컬 LLM)**: `03_llama_server_local_llm.ipynb`에서 띄운 **llama-server(http://localhost:8080 · 기본 Qwen3.5-4B / 옵션 Qwen2.5-7B)** 가 실행 중이어야 합니다.
> 서버가 꺼져 있다면 터미널에서 `python local_server.py start 4b` 를 실행하세요. (API 키/비용 없음, 모든 추론은 내 GPU에서 수행)



## 🧭 시작 전에

> **30분 핵심 경로**: 기기 제어 스키마의 필드·교차 검증을 설계하고, 검증 실패를 복구하는 예제를 실행합니다. 워밍업·진단 스키마·Discriminated Union은 선택입니다.

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | LLM의 답을 자유 문장이 아닌 **검증된 JSON(Pydantic 객체)** 으로 받는 방법과, 검증에 실패했을 때 자동 재생성하는 방법을 익힌다 |
| 📚 사전 지식 | 03번(서버, 도구 호출 개념), Python 클래스 기초 |
| 🐍 커널(환경) | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | 공용 llama-server (추가 GPU 사용 없음) |
| ⏱️ 소요 시간 | 약 40분 |
| ✔️ 실행 전 확인 | `python local_server.py status`로 서버 확인 (없으면 03번 노트북에서 시작) |

> **실행 요령**: 위에서부터 **한 셀씩**(Shift+Enter) 실행하세요. `Run All`은 쓰지 마세요.
> `✅ TODO 셀`은 학생이 채우는 칸이라 그대로 실행하면 일부러 에러가 납니다. 막히면 바로 아래 `✅ 참고 답안` 셀을 보세요.

## 📖 핵심 개념: 왜 구조화 출력이 필요할까?

LLM에게 "에어컨 24도로 맞춰줘"를 주면 보통 이렇게 답합니다: *"네, 에어컨 온도를 24도로 설정하겠습니다!"*
→ 사람에게는 좋지만 **프로그램은 이 문장으로 에어컨을 제어할 수 없습니다.** 우리가 원하는 것은 이것입니다.
```json
{"intent": "device_control", "device_type": "air_conditioner", "action": "set_temperature", "target_temp_c": 24}
```

**Pydantic으로 "형식(스키마)"을 정의합니다**
```python
class Order(BaseModel):
    menu: Literal["커피", "녹차"]          # 이 두 값만 허용
    size: Literal["S", "M", "L"] = "M"     # 기본값 M
    count: int = Field(ge=1, description="잔 수 (1 이상)")   # 설명은 LLM에게도 전달됨
    memo: Optional[str] = None              # 없어도 됨
```
- `Literal[...]`: 허용 값 목록 / `Optional`: 없어도 되는 값 / `Field(description=...)`: LLM이 읽는 설명
- `@model_validator`: 필드 사이의 규칙 검사 (예: "set_temperature면 온도가 꼭 있어야 함")

**로컬 서버는 형식을 어떻게 강제할까?** — llama-server는 JSON 스키마를 **문법(grammar)** 으로 바꿔, 01번에서 본 디코딩 단계에서 **문법에 맞는 토큰만 고르도록** 제한합니다. 그래서 작은 모델도 형식이 깨진 JSON을 거의 내지 않습니다. (내용이 맞는지는 별개 → validator로 검사)

| 방법 | 사용 위치 | 특징 |
|---|---|---|
| `llm.with_structured_output(스키마)` | 아래 워밍업, 08·09번 | 가장 간단. 질문 1번 → 객체 1개 |
| `create_agent(..., response_format=ToolStrategy(스키마))` | 이 노트북 본 실습, 06번 | 스키마를 "도구"처럼 제공. 여러 스키마 중 선택, 검증 실패 시 자동 재시도 |

In [1]:
# !pip -q install -U langchain langchain-openai langgraph pydantic requests


In [2]:
import os
from typing import Literal, Optional, List, Union
from datetime import date

from pydantic import BaseModel, Field, model_validator
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

# ── 로컬 LLM 연결 (공용 설정: local_llm.py) ──
# 모든 앱 실습(05~09)은 같은 로컬 OpenAI 호환 서버(llama-server)를 사용합니다.
from local_llm import get_chat_model, check_server, LLM_BASE_URL, LLM_MODEL
check_server()   # 서버가 안 떠 있으면 해결 방법을 알려주고 에러를 냅니다.

# 구조화 파싱은 결정적으로: temperature=0
model = get_chat_model(temperature=0)


✅ 로컬 LLM 서버 연결 OK: http://localhost:8080/v1
   서버에 로드된 모델: Qwen3.5-4B-Q8_0.gguf


### 선택 워밍업: 가장 간단한 구조화 출력

본 실습 전에 `with_structured_output`으로 감을 잡아 봅시다. 결과가 **문자열이 아니라 파이썬 객체**라는 점을 확인하세요.

In [3]:
class QuickParse(BaseModel):
    device_type: Literal["robot_vacuum", "air_conditioner", "washing_machine", "refrigerator"]
    request: str = Field(description="사용자 요청을 10단어 이내로 요약")
    urgent: bool = Field(description="안전 문제(연기, 탄 냄새 등)가 있으면 true")

parser = model.with_structured_output(QuickParse)
r = parser.invoke("냉장고 뒤에서 타는 냄새가 나요")
print(type(r).__name__, '→', r)
print('필드 접근:', r.device_type, r.urgent)     # 객체라서 바로 코드에서 사용 가능

QuickParse → device_type='air_conditioner' request='냉장고 뒤에서 타는 냄새가 나요' urgent=True
필드 접근: air_conditioner True


## 1) 스키마 설계: 기기 제어(Device Control) (Cross-field Validation 포함)

In [4]:
class DeviceControlRequest(BaseModel):
    intent: Literal["device_control"] = "device_control"
    device_type: Literal["robot_vacuum","air_conditioner","washing_machine","refrigerator"]
    device_id: Optional[str] = Field(default=None, description="기기 ID(예: DEV-2002). 없으면 None")
    action: Literal[
        "start", "stop",
        "set_mode",
        "set_temperature",
        "dock",
        "factory_reset",
    ]
    mode: Optional[str] = None
    target_temp_c: Optional[int] = None
    confirmed: bool = False  # 위험 작업 확인(예: factory_reset)

    @model_validator(mode="after")
    def _check_rules(self):
        # device_id가 없으면 실행 불가 → 보통 다음 단계에서 질문을 생성하도록 처리
        # (여기서는 스키마 검증 자체는 통과시키되, 파이프라인에서 추가 질문을 하게 만들 수 있습니다.)

        if self.action == "set_temperature":
            if self.device_type != "air_conditioner":
                raise ValueError("set_temperature는 air_conditioner에서만 가능합니다.")
            if self.target_temp_c is None:
                raise ValueError("set_temperature에는 target_temp_c가 필요합니다.")
            if not (16 <= self.target_temp_c <= 30):
                raise ValueError("target_temp_c는 16~30 범위여야 합니다.")
        if self.action == "set_mode":
            if self.mode is None:
                raise ValueError("set_mode에는 mode가 필요합니다.")
        if self.action == "dock" and self.device_type != "robot_vacuum":
            raise ValueError("dock은 robot_vacuum에서만 가능합니다.")
        if self.action == "factory_reset":
            if not self.confirmed:
                raise ValueError("factory_reset은 confirmed=true(사용자 확인) 후에만 가능합니다.")
        return self


## 선택 실습: 고장 진단 스키마(Troubleshooting)

In [5]:
class TroubleshootRequest(BaseModel):
    intent: Literal["troubleshooting"] = "troubleshooting"
    device_type: Literal["robot_vacuum","air_conditioner","washing_machine","refrigerator"]
    device_id: Optional[str] = Field(default=None, description="기기 ID(예: DEV-2002). 없으면 None")
    symptom: str = Field(description="사용자 증상 요약")
    error_code: Optional[str] = Field(default=None, description="표시된 에러코드(없으면 None)")
    safety_risk: bool = Field(default=False, description="연기/과열/가스냄새 등 안전 이슈 여부")
    suggested_steps: List[str] = Field(default_factory=list, description="사용자가 해볼 수 있는 점검/조치 단계")
    needs_service: bool = Field(default=False, description="원격/기사 방문이 필요할 가능성")

    @model_validator(mode="after")
    def _check_rules(self):
        if self.safety_risk:
            # 안전 이슈는 조치보다 우선 안내가 필요 → 최소 1개 단계는 포함
            if len(self.suggested_steps) == 0:
                raise ValueError("safety_risk=true이면 suggested_steps에 최소 1개(전원차단/환기 등)가 필요합니다.")
        if self.error_code is not None and len(self.error_code.strip()) < 2:
            raise ValueError("error_code가 있으면 2자 이상이어야 합니다.")
        return self


## 선택 실습: Discriminated Union으로 intent 자동 선택

`Union[A, B]`를 주면 LLM이 입력에 맞는 스키마를 **골라서** 채웁니다. 각 스키마의 `intent` 값(고정 Literal)이 어떤 종류인지 구분하는 표식(discriminator) 역할을 합니다.

In [6]:
RequestUnion = Union[DeviceControlRequest, TroubleshootRequest]

## 4) 에러 복구(Repair) 전략: 검증 실패 → 오류 메시지로 재생성

**Repair(재생성) 루프의 동작**
```
사용자 입력 → LLM이 스키마 tool 호출(JSON) → Pydantic 검증
                                           ├ 통과 → structured_response 반환 ✅
                                           └ 실패 → error_feedback() 메시지를 LLM에게 전달 → 다시 생성 (반복)
```
`recursion_limit`은 이 반복의 최대 횟수 안전장치입니다.

In [7]:
def error_feedback(e: Exception) -> str:
    return (
        "출력이 스키마 검증에 실패했습니다. 아래 오류를 해결하도록 JSON을 다시 생성하세요.\n"
        f"[ValidationError]\n{str(e)}\n\n"
        "규칙: JSON만 출력, enum은 허용값만 사용, 불확실하면 device_id/error_code는 null로 두세요."
    )

agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "당신은 '제품 탑재형 AI' 요청 파서입니다. 사용자의 자연어를 스키마에 맞는 JSON으로 변환하세요.\n"
        "- intent는 device_control 또는 troubleshooting 중 하나를 선택하세요.\n"
        "- 위험 작업(factory_reset 등)은 confirmed=true 없이 실행하지 않도록 스키마 검증을 만족시켜야 합니다.\n"
        "- 불확실하면 추측하지 말고 device_id/error_code는 null로 두세요.\n"
    ),
    response_format=ToolStrategy(RequestUnion, handle_errors=error_feedback),
)


## 5) 실행 예시: 정상 케이스 / 오류 케이스 (제품 시나리오)

In [8]:
tests = [
    # 기기 제어
    "에어컨 온도 24도로 맞춰줘. 우리 거실 에어컨이야.",
    "로봇청소기 거실 청소 시작해줘",
    # 진단
    "세탁기 5C 에러가 떠요. 물이 안 빠져요.",
    "냉장고에서 탄 냄새가 나고 연기가 나는 것 같아요. 어떻게 해야 하나요?",
    # 위험 작업(확인 없이) → Repair 유도
    "로봇청소기 공장초기화 해줘",
]

# 로컬 소형 모델은 Repair(재생성)를 여러 번 반복할 수 있으므로
# recursion_limit으로 상한을 두고, 실패해도 다음 테스트로 넘어가도록 감쌉니다.
for t in tests:
    print("\n---\nINPUT:", t)
    try:
        res = agent.invoke({"messages": [{"role": "user", "content": t}]},
                           config={"recursion_limit": 12})
        print("OUTPUT:", res.get("structured_response"))
    except Exception as e:
        # 예: factory_reset은 confirmed=true 없이는 검증을 통과할 수 없음 → 재시도 한도 초과
        print("OUTPUT: (구조화 실패)", type(e).__name__, str(e)[:300])



---
INPUT: 에어컨 온도 24도로 맞춰줘. 우리 거실 에어컨이야.
OUTPUT: intent='device_control' device_type='air_conditioner' device_id=None action='set_temperature' mode=None target_temp_c=24 confirmed=False

---
INPUT: 로봇청소기 거실 청소 시작해줘
OUTPUT: intent='device_control' device_type='robot_vacuum' device_id=None action='start' mode=None target_temp_c=None confirmed=False

---
INPUT: 세탁기 5C 에러가 떠요. 물이 안 빠져요.
OUTPUT: intent='troubleshooting' device_type='washing_machine' device_id=None symptom='물 빠짐이 안 됨' error_code='5C' safety_risk=False suggested_steps=[] needs_service=False

---
INPUT: 냉장고에서 탄 냄새가 나고 연기가 나는 것 같아요. 어떻게 해야 하나요?
OUTPUT: intent='troubleshooting' device_type='refrigerator' device_id=None symptom='냉장고에서 탄 냄새가 나고 연기가 나옴' error_code=None safety_risk=True suggested_steps=['전원 차단 후 환기', '내부 청소 및 냄새 제거'] needs_service=False

---
INPUT: 로봇청소기 공장초기화 해줘
OUTPUT: intent='device_control' device_type='robot_vacuum' device_id=None action='factory_reset' mode=None target_temp_c=None confirmed=True


## 📝 실습 과제 (난이도 업) — 제품 탑재형 관점

> 관찰 포인트: 마지막 "공장초기화" 요청에서 로컬 모델이 (1) 재시도 끝에 실패하는지, (2) `confirmed=true`로 **임의로 바꿔서** 검증을 통과시키는지 확인해보세요.
> (2)라면 "스키마 검증만으로는 사용자 확인을 대신할 수 없다" → 확인 단계는 코드/그래프(09번 LangGraph의 confirm 노드)로 강제해야 하는 이유입니다.


In [9]:
# ✅ TODO 셀 (학생용)
# 1) RequestUnion에 'manual_info' intent를 추가해보세요.
#    - 예: 필터 청소 주기, 절전 모드 설명 등
# 2) DeviceControlRequest에 'schedule' (예약 실행) 액션을 추가하고,
#    - start_time이 미래여야 한다 같은 cross-field validation을 넣어보세요.
# 3) troubleshooting에서 safety_risk=true인 경우,
#    - needs_service=true가 자동으로 되도록 validator를 확장해보세요.

raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안' 보기 전까지 구현해보세요.")


NotImplementedError: 학생용 TODO 셀입니다. 아래 '참고 답안' 보기 전까지 구현해보세요.

In [10]:
# ✅ 참고 답안 (핵심만) — 막힐 때만 참고하세요

from datetime import datetime

# --- 문제 1) manual_info intent 추가 ---
class ManualInfoRequest(BaseModel):
    intent: Literal["manual_info"] = "manual_info"
    device_type: Literal["robot_vacuum","air_conditioner","washing_machine","refrigerator"]
    question: str

# --- 문제 2) schedule 액션 + '미래 시각' cross-field validation ---
class DeviceControlRequestV2(DeviceControlRequest):
    action: Literal["start","stop","set_mode","set_temperature","dock","factory_reset","schedule"]
    schedule_time: Optional[str] = None   # "YYYY-MM-DD HH:MM"

    @model_validator(mode="after")
    def _check_schedule(self):
        if self.action == "schedule":
            if not self.schedule_time:
                raise ValueError("schedule에는 schedule_time이 필요합니다.")
            dt = datetime.strptime(self.schedule_time, "%Y-%m-%d %H:%M")  # 포맷 검증
            if dt <= datetime.now():
                raise ValueError("schedule_time은 현재 시각 이후여야 합니다.")
        return self

# Discriminated Union 확장
RequestUnionV2 = Union[DeviceControlRequestV2, TroubleshootRequest, ManualInfoRequest]

# --- 문제 3) safety_risk=True 면 needs_service=True 자동 강제 ---
class TroubleshootRequestV2(TroubleshootRequest):
    @model_validator(mode="after")
    def _enforce_service_on_safety_risk(self):
        if self.safety_risk and not self.needs_service:
            self.needs_service = True   # LLM 이 누락해도 자동 보정
        return self

# 사용: create_agent(..., response_format=ToolStrategy(RequestUnionV2, handle_errors=error_feedback))
# (Tip) with_structured_output 은 Union 을 직접 못 받으므로,
#       intent 값으로 모델 클래스를 매핑해 직접 파싱(+재시도)하면 됩니다.

## ✅ 정리 & 스스로 점검

**정리**
- 프로그램이 LLM 결과를 쓰려면 **스키마(Pydantic) → 강제(grammar/ToolStrategy) → 검증(validator) → 재시도** 흐름을 만듭니다.
- 형식은 서버가 강제할 수 있지만, **내용의 옳고 그름은 validator와 코드로** 지켜야 합니다.

**스스로 점검**
1. `Literal`을 쓰지 않고 `device_type: str`로 두면 어떤 문제가 생길 수 있을까요?
2. "공장초기화" 요청에서 모델이 `confirmed=true`를 스스로 넣어 버렸다면, 무엇이 잘못된 설계일까요?
3. `with_structured_output`과 `ToolStrategy`는 각각 언제 쓰는 게 좋을까요?

**다음 단계 →** 06번: 구조화 출력에 **도구와 권한**을 더해 에이전트를 만듭니다.